Week 8 Capstone: Notebook 1 - Data Cleaning Pipeline
Cell-by-Cell Code Guide

In [3]:
# Import libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported")


✅ Libraries imported


In [6]:
# Load raw datasets
sales_raw = pd.read_csv(r"C:\Users\ravis\Downloads\sales_data.csv")
churn_raw = pd.read_csv(r"C:\Users\ravis\Downloads\customer_churn.csv")

print(f"Sales shape: {sales_raw.shape}")
print(f"Churn shape: {churn_raw.shape}")
print("\nSales sample:")
print(sales_raw.head())
print("\nChurn sample:")
print(churn_raw.head())


Sales shape: (100, 7)
Churn shape: (500, 9)

Sales sample:
         Date     Product  Quantity  Price Customer_ID Region  Total_Sales
0  01-01-2024       Phone         7  37300     CUST001   East       261100
1  02-01-2024  Headphones         4  15406     CUST002  North        61624
2  03-01-2024       Phone         2  21746     CUST003   West        43492
3  04-01-2024  Headphones         1  30895     CUST004   East        30895
4  05-01-2024      Laptop         8  39835     CUST005  North       318680

Churn sample:
  CustomerID  Tenure  MonthlyCharges  TotalCharges        Contract  \
0     C00001       6              64          1540        One year   
1     C00002      21             113          1753  Month-to-month   
2     C00003      27              31          1455        Two year   
3     C00004      53              29          7150  Month-to-month   
4     C00005      16             185          1023        One year   

      PaymentMethod PaperlessBilling  SeniorCitizen  Ch

In [7]:
# Standardize column names
sales_df = sales_raw.copy()
sales_df.columns = sales_df.columns.str.strip().str.lower().str.replace(' ', '_')

# Parse dates (DD-MM-YYYY format)
sales_df['date'] = pd.to_datetime(sales_df['date'], dayfirst=True, errors='coerce')
sales_df['month'] = sales_df['date'].dt.to_period('M').astype(str)

# Numeric conversion
for col in ['quantity', 'price', 'total_sales']:
    if col in sales_df.columns:
        sales_df[col] = pd.to_numeric(sales_df[col], errors='coerce')

print(f"Columns: {list(sales_df.columns)}")
print(f"\nData types:\n{sales_df.dtypes}")



Columns: ['date', 'product', 'quantity', 'price', 'customer_id', 'region', 'total_sales', 'month']

Data types:
date           datetime64[ns]
product                object
quantity                int64
price                   int64
customer_id            object
region                 object
total_sales             int64
month                  object
dtype: object


In [8]:
# Standardize column names
churn_df = churn_raw.copy()
churn_df.columns = churn_df.columns.str.strip().str.lower().str.replace(' ', '_')

# Convert churn to binary
churn_df['churn_flag'] = churn_df['churn'].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)

# Numeric conversion
for col in ['tenure', 'monthlycharges', 'totalcharges']:
    if col in churn_df.columns:
        churn_df[col] = pd.to_numeric(churn_df[col], errors='coerce')

print(f"Columns: {list(churn_df.columns)}")
print(f"\nChurn rate: {churn_df['churn_flag'].mean()*100:.1f}%")



Columns: ['customerid', 'tenure', 'monthlycharges', 'totalcharges', 'contract', 'paymentmethod', 'paperlessbilling', 'seniorcitizen', 'churn', 'churn_flag']

Churn rate: 0.0%


In [9]:
# Sales cleaning
sales_clean = sales_df.dropna(subset=['date', 'quantity', 'price', 'total_sales'])
print(f"Sales: {len(sales_clean)}/{len(sales_df)} records ({len(sales_clean)/len(sales_df)*100:.1f}%)")

# Churn cleaning
churn_clean = churn_df.dropna(subset=['tenure', 'monthlycharges', 'churn_flag'])
print(f"Churn: {len(churn_clean)}/{len(churn_df)} records ({len(churn_clean)/len(churn_df)*100:.1f}%)")

# Missing values
print(f"\nMissing values in sales: {sales_clean.isnull().sum().sum()}")
print(f"Missing values in churn: {churn_clean.isnull().sum().sum()}")


Sales: 100/100 records (100.0%)
Churn: 500/500 records (100.0%)

Missing values in sales: 0
Missing values in churn: 0


In [16]:
# Sales statistics
print("📊 SALES STATISTICS")
print(sales_clean[['quantity', 'price', 'total_sales']].describe().round(2))

print("\n📊 CHURN STATISTICS")
print(churn_clean[['tenure', 'monthlycharges', 'churn_flag']].describe().round(2))


📊 SALES STATISTICS
       quantity     price  total_sales
count    100.00    100.00       100.00
mean       4.78  25808.51    123650.48
std        2.59  13917.63    100161.09
min        1.00   1308.00      6540.00
25%        2.75  14965.25     39517.50
50%        5.00  24192.00     97955.50
75%        7.00  38682.25    175792.50
max        9.00  49930.00    373932.00

📊 CHURN STATISTICS
       tenure  monthlycharges  churn_flag
count  500.00          500.00       500.0
mean    36.53          113.64         0.0
std     20.67           51.80         0.0
min      1.00           20.00         0.0
25%     19.00           67.00         0.0
50%     37.00          115.00         0.0
75%     54.00          158.00         0.0
max     71.00          199.00         0.0


In [17]:
# Save cleaned datasets
sales_clean.to_csv(r"C:\Users\ravis\Downloads\sales_data.csv", index=False)
churn_clean.to_csv(r"C:\Users\ravis\Downloads\customer_churn.csv", index=False)

print("✅ Clean datasets saved!")
print(f"   sales_data_clean.csv ({len(sales_clean)} records)")
print(f"   churn_data_clean.csv ({len(churn_clean)} records)")


✅ Clean datasets saved!
   sales_data_clean.csv (100 records)
   churn_data_clean.csv (500 records)


In [18]:
# Quick verification
print("\n" + "="*60)
print("✅ DATA CLEANING COMPLETE")
print("="*60)
print(f"Sales Dataset:")
print(f"  • Records: {len(sales_clean)}")
print(f"  • Columns: {len(sales_clean.columns)}")
print(f"  • Missing: {sales_clean.isnull().sum().sum()}")
print(f"\nChurn Dataset:")
print(f"  • Records: {len(churn_clean)}")
print(f"  • Columns: {len(churn_clean.columns)}")
print(f"  • Missing: {churn_clean.isnull().sum().sum()}")
print(f"\nReady for: EDA & Analysis ✅")



✅ DATA CLEANING COMPLETE
Sales Dataset:
  • Records: 100
  • Columns: 8
  • Missing: 0

Churn Dataset:
  • Records: 500
  • Columns: 10
  • Missing: 0

Ready for: EDA & Analysis ✅
